# Worst-rounds human-vs-AI audit

Analysis-only audit of E2025 Matchdays 23, 1, 12, 5, and 38. This notebook reuses frozen prediction, market, rules, and replay artifacts; it does not train or modify any model or optimizer.

In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import pickle
import sys

import duckdb
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.strategy.coach import build_coach_feature_history, coach_market_for_matchday
from src.strategy.engine import replay_actual_round
from src.strategy.historical import load_historical_market
from src.strategy.optimizer import (
    entity_ids_for_result,
    optimize_deterministic,
    optimize_roles_for_roster,
)
from src.strategy.rules import Lineup, load_rules, score_lineup, validate_lineup

DATABASE_PATH = PROJECT_ROOT / "data/db/euroleague.duckdb"
PREDICTIONS_PATH = PROJECT_ROOT / "data/derived/phase7/research/e2025_full_market_predictions.parquet"
BACKTEST_PATH = PROJECT_ROOT / "data/derived/phase7/research/e2025_strategy_backtest.parquet"
CUTOFF_AUDIT_PATH = PROJECT_ROOT / "data/derived/phase7/research/e2025_replay_cutoff_audit.json"
COMPARISON_PATH = PROJECT_ROOT / "data/benchmarks/e2025_ai_human_round_comparison.csv"
COACH_MODEL_PATH = PROJECT_ROOT / "data/derived/phase7/research/coach_distribution_model.pkl"

WORST_MATCHDAYS = (23, 1, 12, 5, 38)
STRATEGY = "FULL_DYNAMIC_STOCHASTIC"
rules = load_rules()
print(rules.ruleset_version, WORST_MATCHDAYS)

elfc_classic_e2025_v1 (23, 1, 12, 5, 38)


In [2]:
with duckdb.connect() as connection:
    predictions = connection.execute(
        "SELECT * FROM read_parquet(?) WHERE round_number IN (SELECT unnest(?))",
        [str(PREDICTIONS_PATH), list(WORST_MATCHDAYS)],
    ).fetchdf()
    backtest = connection.execute(
        "SELECT * FROM read_parquet(?) WHERE strategy = ?",
        [str(BACKTEST_PATH), STRATEGY],
    ).fetchdf()

comparison = pd.read_csv(COMPARISON_PATH)
cutoff_payload = json.loads(CUTOFF_AUDIT_PATH.read_text())
cutoff_audit = pd.DataFrame(cutoff_payload["matchdays"])
with COACH_MODEL_PATH.open("rb") as handle:
    coach_model = pickle.load(handle)

selected_columns = [
    "player_id", "entity_id", "name", "position", "team_id", "credits", "turn",
    "expected_fp", "actual_fp", "actual_minutes", "played", "p10_fp", "p50_fp",
    "p90_fp", "phase4b_expected_minutes", "expected_minutes_after_availability",
]
print(predictions.shape)
print([column for column in selected_columns if column in predictions.columns])
print(backtest.loc[backtest.matchday.isin(WORST_MATCHDAYS), [
    "matchday", "actual_final_score", "available_budget", "total_credits",
    "bank_after", "trades", "trade_limit",
]].sort_values("matchday").to_string(index=False))

(1519, 300)
['player_id', 'entity_id', 'name', 'position', 'team_id', 'credits', 'turn', 'expected_fp', 'actual_fp', 'actual_minutes', 'played', 'p10_fp', 'p50_fp', 'p90_fp']
 matchday  actual_final_score  available_budget  total_credits  bank_after  trades  trade_limit
        1              112.25             100.0          100.0         0.0      11          NaN
        5              143.20             105.3          105.3         0.0       3          4.0
       12              122.25             115.1          114.2         0.9       4          4.0
       23              125.25             131.8          131.7         0.1       4          4.0
       38              168.45             149.1          149.1         0.0       4          4.0


In [3]:
def pipe_ids(value: str) -> tuple[str, ...]:
    return tuple(item for item in str(value).split("|") if item)


def role_multiplier(lineup: Lineup, player_id: str) -> float:
    if player_id in lineup.starters:
        multiplier = 1.0
    elif player_id == lineup.sixth_man:
        multiplier = rules.sixth_man_multiplier
    else:
        multiplier = rules.bench_multiplier
    if player_id == lineup.captain:
        multiplier += rules.captain_multiplier - 1.0
    return float(multiplier)


def actual_score(result, player_pool, coach_pool) -> float:
    player_scores = dict(zip(player_pool.player_id.astype(str), player_pool.actual_fp.astype(float)))
    coach_score = float(
        coach_pool.loc[
            coach_pool.coach_id.astype(str).eq(result.lineup.coach_id), "actual_score"
        ].iloc[0]
    )
    return score_lineup(result.lineup, player_scores, coach_score, rules)


with duckdb.connect() as connection:
    identity = connection.execute(
        "SELECT DISTINCT player_id, entity_id FROM read_parquet(?)",
        [str(PREDICTIONS_PATH)],
    ).fetchdf()
assert identity.groupby("player_id")["entity_id"].nunique().le(1).all()
player_to_entity = dict(identity.itertuples(index=False, name=None))

coach_history = build_coach_feature_history(DATABASE_PATH)
round_contexts = {}
attribution_rows = []
player_detail_frames = []

for matchday in WORST_MATCHDAYS:
    player_pool = predictions.loc[predictions.round_number.eq(matchday)].copy()
    player_pool["player_id"] = player_pool.player_id.astype(str)
    player_pool["entity_id"] = player_pool.entity_id.astype(str)
    player_pool["position"] = player_pool.position.astype(str)
    player_pool["expected_fp"] = pd.to_numeric(player_pool.expected_fp, errors="raise")
    player_pool["actual_fp"] = pd.to_numeric(player_pool.actual_fp, errors="raise")
    player_pool["credits"] = pd.to_numeric(player_pool.credits, errors="raise")
    player_pool["predicted_position_rank"] = (
        player_pool.groupby("position")["expected_fp"]
        .rank(method="min", ascending=False)
        .astype(int)
    )
    player_pool["actual_round_rank"] = (
        player_pool["actual_fp"].rank(method="min", ascending=False).astype(int)
    )

    market = load_historical_market("E2025", matchday, database_path=DATABASE_PATH)
    coach_pool = coach_market_for_matchday(
        coach_history, coach_model, "E2025", matchday, market
    )
    row = backtest.loc[backtest.matchday.eq(matchday)].iloc[0]
    initial = Lineup(
        pipe_ids(row.roster),
        str(row.coach_id),
        frozenset(pipe_ids(row.starters)),
        str(row.sixth_man),
        str(row.captain),
    )
    selected_coach = coach_pool.loc[
        coach_pool.coach_id.astype(str).eq(initial.coach_id)
    ].iloc[0]
    coach_actual = float(selected_coach.actual_score)
    replay = replay_actual_round(initial, player_pool, coach_actual, rules)

    roster_pool = player_pool.loc[player_pool.player_id.isin(initial.player_ids)].copy()
    player_records = {
        str(p.player_id): {
            "position": str(p.position),
            "team_id": str(p.team_id),
            "credits": float(p.credits),
        }
        for p in roster_pool.itertuples(index=False)
    }
    coach_record = {
        "team_id": str(selected_coach.team_id),
        "credits": float(selected_coach.credits),
    }
    budget = float(row.available_budget)
    validate_lineup(initial, player_records, coach_record, rules, budget=budget)
    validate_lineup(replay.final_lineup, player_records, coach_record, rules, budget=budget)
    assert np.isclose(replay.final_score, row.actual_final_score)
    assert np.isclose(replay.static_score, row.actual_static_score)

    same_roster_oracle = optimize_roles_for_roster(
        player_pool, coach_pool, rules, initial.player_ids, initial.coach_id,
        objective="actual_fp",
    )
    same_roster_oracle_score = actual_score(
        same_roster_oracle, player_pool, coach_pool
    )

    if matchday == 1:
        previous_entities = frozenset()
        trade_limit = None
    else:
        previous = backtest.loc[backtest.matchday.eq(matchday - 1)].iloc[0]
        previous_entities = frozenset(
            [player_to_entity[player] for player in pipe_ids(previous.roster)]
            + [str(previous.coach_id)]
        )
        trade_limit = int(row.trade_limit)

    oracle_limited = optimize_deterministic(
        player_pool, coach_pool, rules, budget=budget,
        player_objective="actual_fp", coach_objective="actual_score",
        previous_entity_ids=previous_entities, trade_limit=trade_limit,
    )
    oracle_no_transfer = optimize_deterministic(
        player_pool, coach_pool, rules, budget=budget,
        player_objective="actual_fp", coach_objective="actual_score",
    )
    oracle_no_budget = optimize_deterministic(
        player_pool, coach_pool, rules, budget=10_000.0,
        player_objective="actual_fp", coach_objective="actual_score",
    )
    deterministic_expected = optimize_deterministic(
        player_pool, coach_pool, rules, budget=budget,
        previous_entity_ids=previous_entities, trade_limit=trade_limit,
    )
    deterministic_expected_actual = actual_score(
        deterministic_expected, player_pool, coach_pool
    )

    expected_scores = dict(zip(player_pool.player_id, player_pool.expected_fp))
    actual_scores = dict(zip(player_pool.player_id, player_pool.actual_fp))
    predicted_initial_score = score_lineup(
        initial, expected_scores, float(selected_coach.expected_score), rules
    )
    actual_initial_score = score_lineup(
        initial, actual_scores, coach_actual, rules
    )

    detail = roster_pool[[
        "player_id", "entity_id", "name", "position", "team_id", "credits", "turn",
        "expected_fp", "actual_fp", "actual_minutes", "played", "p10_fp", "p50_fp",
        "p90_fp", "baseline_expected_minutes", "predicted_position_rank",
        "actual_round_rank",
    ]].copy()
    detail.insert(0, "matchday", matchday)
    detail["initial_role"] = detail.player_id.map(
        lambda player: "CAPTAIN" if player == initial.captain
        else "STARTER" if player in initial.starters
        else "SIXTH" if player == initial.sixth_man
        else "BENCH"
    )
    detail["final_role"] = detail.player_id.map(
        lambda player: "CAPTAIN" if player == replay.final_lineup.captain
        else "STARTER" if player in replay.final_lineup.starters
        else "SIXTH" if player == replay.final_lineup.sixth_man
        else "BENCH"
    )
    detail["final_multiplier"] = detail.player_id.map(
        lambda player: role_multiplier(replay.final_lineup, player)
    )
    detail["fp_error"] = detail.actual_fp - detail.expected_fp
    detail["minutes_error"] = (
        detail.actual_minutes - detail.baseline_expected_minutes
    )
    detail["weighted_fp_error"] = detail.final_multiplier * detail.fp_error
    detail["tail_outcome"] = np.select(
        [detail.actual_fp < detail.p10_fp, detail.actual_fp > detail.p90_fp],
        ["BELOW_P10", "ABOVE_P90"],
        default="WITHIN_P10_P90",
    )
    player_detail_frames.append(detail)

    attribution_rows.append({
        "matchday": matchday,
        "ai_score": float(row.actual_final_score),
        "top10_average": float(
            comparison.loc[comparison.matchday.eq(matchday), "top10_average"].iloc[0]
        ),
        "gap_vs_top10": float(
            comparison.loc[
                comparison.matchday.eq(matchday), "gap_vs_top10_average"
            ].iloc[0]
        ),
        "predicted_initial_score": float(predicted_initial_score),
        "actual_initial_score": float(actual_initial_score),
        "selected_prediction_residual": float(actual_initial_score - predicted_initial_score),
        "same_roster_hindsight_score": float(same_roster_oracle_score),
        "captain_starter_hindsight_loss": float(
            same_roster_oracle_score - row.actual_final_score
        ),
        "constrained_hindsight_oracle": float(oracle_limited.objective_value),
        "selection_hindsight_loss": float(
            oracle_limited.objective_value - same_roster_oracle_score
        ),
        "transfer_constraint_upper_bound": float(
            oracle_no_transfer.objective_value - oracle_limited.objective_value
        ),
        "budget_constraint_upper_bound": float(
            oracle_no_budget.objective_value - oracle_no_transfer.objective_value
        ),
        "unconstrained_hindsight_oracle": float(oracle_no_budget.objective_value),
        "deterministic_expected_actual": float(deterministic_expected_actual),
        "stochastic_minus_deterministic_actual": float(
            row.actual_final_score - deterministic_expected_actual
        ),
        "available_budget": budget,
        "unused_credits": float(row.bank_after),
        "transfers": int(row.trades),
        "trade_limit": trade_limit,
        "coach": str(selected_coach["name"]),
        "coach_predicted": float(selected_coach.expected_score),
        "coach_actual": coach_actual,
        "initial_captain": str(
            roster_pool.loc[roster_pool.player_id.eq(initial.captain), "name"].iloc[0]
        ),
        "final_captain": str(
            roster_pool.loc[
                roster_pool.player_id.eq(replay.final_lineup.captain), "name"
            ].iloc[0]
        ),
        "captain_switches": replay.captain_switches,
        "substitutions": replay.substitutions,
        "captured_actual_top10": int(
            player_pool.nlargest(10, "actual_fp").player_id.isin(initial.player_ids).sum()
        ),
    })
    round_contexts[matchday] = {
        "player_pool": player_pool,
        "coach_pool": coach_pool,
        "row": row,
        "initial": initial,
        "replay": replay,
        "previous_entities": previous_entities,
        "trade_limit": trade_limit,
        "oracle_limited": oracle_limited,
        "oracle_no_transfer": oracle_no_transfer,
        "oracle_no_budget": oracle_no_budget,
    }

attribution = pd.DataFrame(attribution_rows).sort_values(
    "matchday"
).reset_index(drop=True)
selected_details = pd.concat(player_detail_frames, ignore_index=True)
print(attribution.round(2).to_string(index=False))

 matchday  ai_score  top10_average  gap_vs_top10  predicted_initial_score  actual_initial_score  selected_prediction_residual  same_roster_hindsight_score  captain_starter_hindsight_loss  constrained_hindsight_oracle  selection_hindsight_loss  transfer_constraint_upper_bound  budget_constraint_upper_bound  unconstrained_hindsight_oracle  deterministic_expected_actual  stochastic_minus_deterministic_actual  available_budget  unused_credits  transfers  trade_limit             coach  coach_predicted  coach_actual  initial_captain    final_captain  captain_switches  substitutions  captured_actual_top10
        1    112.25         221.06       -108.80                   108.25                 70.00                        -38.25                       117.30                            5.05                        261.05                    143.75                             0.00                          23.60                          284.65                          97.05                         

In [4]:
def force_candidate_feasible(
    player_pool: pd.DataFrame,
    coach_pool: pd.DataFrame,
    candidate_id: str,
    *,
    budget: float,
    previous_entities: frozenset[str],
    trade_limit: int | None,
) -> bool:
    diagnostic_pool = player_pool.copy()
    diagnostic_pool["_force_candidate"] = diagnostic_pool["expected_fp"] + np.where(
        diagnostic_pool.player_id.eq(candidate_id), 1_000_000.0, 0.0
    )
    try:
        result = optimize_deterministic(
            diagnostic_pool, coach_pool, rules, budget=budget,
            player_objective="_force_candidate",
            previous_entity_ids=previous_entities,
            trade_limit=trade_limit,
        )
    except ValueError:
        return False
    return candidate_id in result.lineup.player_ids


missed_rows = []
underperformer_rows = []

for matchday in WORST_MATCHDAYS:
    context = round_contexts[matchday]
    pool = context["player_pool"]
    selected_ids = set(context["initial"].player_ids)
    budget = float(context["row"].available_budget)
    previous_entities = context["previous_entities"]
    trade_limit = context["trade_limit"]

    missed = pool.loc[~pool.player_id.isin(selected_ids)].nlargest(6, "actual_fp")
    for candidate in missed.itertuples(index=False):
        exact = force_candidate_feasible(
            pool, context["coach_pool"], str(candidate.player_id),
            budget=budget, previous_entities=previous_entities,
            trade_limit=trade_limit,
        )
        no_transfer = force_candidate_feasible(
            pool, context["coach_pool"], str(candidate.player_id),
            budget=budget, previous_entities=frozenset(), trade_limit=None,
        )
        no_budget = force_candidate_feasible(
            pool, context["coach_pool"], str(candidate.player_id),
            budget=10_000.0, previous_entities=previous_entities,
            trade_limit=trade_limit,
        )
        unconstrained = force_candidate_feasible(
            pool, context["coach_pool"], str(candidate.player_id),
            budget=10_000.0, previous_entities=frozenset(), trade_limit=None,
        )
        if exact:
            blocker = "NONE_SELECTABLE"
        elif no_transfer and not no_budget:
            blocker = "TRANSFER_CONSTRAINT"
        elif no_budget and not no_transfer:
            blocker = "BUDGET_CONSTRAINT"
        elif no_transfer and no_budget:
            blocker = "BUDGET_OR_TRANSFER"
        elif unconstrained:
            blocker = "COMBINED_BUDGET_TRANSFER"
        else:
            blocker = "STRUCTURAL_ROSTER_CONSTRAINT"

        tail = (
            "ABOVE_P90" if candidate.actual_fp > candidate.p90_fp
            else "BELOW_P10" if candidate.actual_fp < candidate.p10_fp
            else "WITHIN_P10_P90"
        )
        missed_rows.append({
            "matchday": matchday,
            "name": candidate.name,
            "position": candidate.position,
            "credits": float(candidate.credits),
            "expected_fp": float(candidate.expected_fp),
            "actual_fp": float(candidate.actual_fp),
            "fp_error": float(candidate.actual_fp - candidate.expected_fp),
            "expected_minutes": float(candidate.baseline_expected_minutes),
            "actual_minutes": float(candidate.actual_minutes),
            "minutes_error": float(
                candidate.actual_minutes - candidate.baseline_expected_minutes
            ),
            "predicted_position_rank": int(candidate.predicted_position_rank),
            "actual_round_rank": int(candidate.actual_round_rank),
            "played": bool(candidate.played),
            "tail_outcome": tail,
            "legal_with_actual_constraints": exact,
            "legal_without_transfer_limit": no_transfer,
            "legal_without_budget_limit": no_budget,
            "legal_without_both": unconstrained,
            "blocker": blocker,
        })

    underperformers = (
        selected_details.loc[selected_details.matchday.eq(matchday)]
        .sort_values(["weighted_fp_error", "fp_error"])
        .head(5)
    )
    underperformer_rows.extend(underperformers.to_dict("records"))

missed_high_scorers = pd.DataFrame(missed_rows)
selected_underperformers = pd.DataFrame(underperformer_rows)

for matchday in WORST_MATCHDAYS:
    print()
    print("MD", matchday, "SELECTED UNDERPERFORMERS")
    print(selected_underperformers.loc[
        selected_underperformers.matchday.eq(matchday),
        [
            "name", "position", "credits", "initial_role", "final_role",
            "expected_fp", "actual_fp", "fp_error", "baseline_expected_minutes",
            "actual_minutes", "minutes_error", "tail_outcome",
        ],
    ].round(2).to_string(index=False))
    print("MD", matchday, "MISSED HIGH SCORERS")
    print(missed_high_scorers.loc[
        missed_high_scorers.matchday.eq(matchday),
        [
            "name", "position", "credits", "expected_fp", "actual_fp", "fp_error",
            "expected_minutes", "actual_minutes", "minutes_error",
            "predicted_position_rank", "actual_round_rank", "tail_outcome",
            "legal_with_actual_constraints", "blocker",
        ],
    ].round(2).to_string(index=False))


MD 23 SELECTED UNDERPERFORMERS
              name position  credits initial_role final_role  expected_fp  actual_fp  fp_error  baseline_expected_minutes  actual_minutes  minutes_error   tail_outcome
     Kendrick Nunn    GUARD     17.0      STARTER    STARTER        20.36        0.0    -20.36                      28.68            0.00         -28.68      BELOW_P10
      Wade Baldwin    GUARD     14.6        SIXTH      SIXTH        17.55        0.0    -17.55                      25.45            0.00         -25.45      BELOW_P10
        Mike James    GUARD     17.1      STARTER    STARTER        20.28       13.0     -7.28                      29.75           35.15           5.40 WITHIN_P10_P90
Shaquille Harrison  FORWARD      6.9        BENCH      BENCH         8.37        2.0     -6.37                      19.17           21.55           2.38 WITHIN_P10_P90
         Dan Oturu   CENTER     14.0      STARTER    STARTER        18.22       15.4     -2.82                      23.56       

## Recoverable human lineup evidence

BasketStories publishes the winning team and score for each round, but the round articles expose only partial lineup prose—not full roster cards. The table below treats only explicitly named players/roles as human evidence. An ambiguous surname is kept ambiguous rather than guessed.

In [5]:
human_sources = pd.DataFrame([
    {
        "matchday": 1,
        "manager": "Christos Gouziotis / King of basketball",
        "winner_score": 226.85,
        "source_url": "https://www.basketstories.net/article.php?p=private_league_post_5564",
        "published_lineup_evidence": "No player names; article says every starter scored at least 25 FP.",
    },
    {
        "matchday": 5,
        "manager": "Andreas Fountas / Avengers",
        "winner_score": 233.15,
        "source_url": "https://www.basketstories.net/article.php?p=private_league_post_5616",
        "published_lineup_evidence": (
            "Both Baldwins (Kamar and Wade); Jordan Nwora and Dan Oturu explicitly "
            "described as bench; an unnamed Olympiacos trio."
        ),
    },
    {
        "matchday": 12,
        "manager": "Nikolaos Dimopoulos / Lion City Greens",
        "winner_score": 214.30,
        "source_url": "https://www.basketstories.net/article.php?p=private_league_post_5713",
        "published_lineup_evidence": (
            "Kendrick Nunn, Kenneth Faried, Ercan Osmani and Nick Weiler-Babb "
            "explicitly named; roles/captain not published."
        ),
    },
    {
        "matchday": 23,
        "manager": "Dimitrios Kondouris / Melbourniakos 13",
        "winner_score": 246.90,
        "source_url": "https://www.basketstories.net/article.php?p=private_league_post_5854",
        "published_lineup_evidence": (
            "Kameron Taylor and coach Pedro Martínez named; Talen Horton-Tucker "
            "explicitly described as bench."
        ),
    },
    {
        "matchday": 38,
        "manager": "Vag Vent / Nigel Williams-Vagoss",
        "winner_score": 258.35,
        "source_url": "https://www.basketstories.net/article.php?p=private_league_post_6025",
        "published_lineup_evidence": (
            "Moses Wright explicitly captain; Matthew Strazel and Jaron Blossomgame "
            "named; 'Edwards' named without enough detail to distinguish Carsen/Kessler."
        ),
    },
]).sort_values("matchday")

human_player_clues = {
    5: [
        ("Kamar Baldwin", "named"),
        ("Wade Baldwin", "named"),
        ("Jordan Nwora", "bench"),
        ("Dan Oturu", "bench"),
    ],
    12: [
        ("Kendrick Nunn", "named"),
        ("Kenneth Faried", "named"),
        ("Ercan Osmani", "named"),
        ("Nick Weiler-Babb", "named"),
    ],
    23: [
        ("Kameron Taylor", "named"),
        ("Talen Horton-Tucker", "bench"),
    ],
    38: [
        ("Moses Wright", "captain"),
        ("Matthew Strazel", "named"),
        ("Jaron Blossomgame", "named"),
        ("Carsen Edwards", "ambiguous Edwards candidate"),
        ("Kessler Edwards", "ambiguous Edwards candidate"),
    ],
}

human_clue_rows = []
for matchday, clues in human_player_clues.items():
    pool = round_contexts[matchday]["player_pool"]
    selected = selected_details.loc[selected_details.matchday.eq(matchday)]
    for player_name, published_role in clues:
        matches = pool.loc[pool.name.eq(player_name)]
        if matches.empty:
            human_clue_rows.append({
                "matchday": matchday,
                "name": player_name,
                "published_role": published_role,
                "available": False,
            })
            continue
        player = matches.iloc[0]
        selected_match = selected.loc[selected.name.eq(player_name)]
        human_clue_rows.append({
            "matchday": matchday,
            "name": player_name,
            "published_role": published_role,
            "available": True,
            "ai_selected": not selected_match.empty,
            "ai_initial_role": (
                str(selected_match.initial_role.iloc[0]) if not selected_match.empty else None
            ),
            "ai_final_role": (
                str(selected_match.final_role.iloc[0]) if not selected_match.empty else None
            ),
            "credits": float(player.credits),
            "predicted_fp": float(player.expected_fp),
            "actual_fp": float(player.actual_fp),
            "fp_error": float(player.actual_fp - player.expected_fp),
            "predicted_position_rank": int(player.predicted_position_rank),
            "actual_round_rank": int(player.actual_round_rank),
            "predicted_minutes": float(player.baseline_expected_minutes),
            "actual_minutes": float(player.actual_minutes),
            "tail_outcome": (
                "BELOW_P10" if player.actual_fp < player.p10_fp
                else "ABOVE_P90" if player.actual_fp > player.p90_fp
                else "WITHIN_P10_P90"
            ),
        })

human_clues = pd.DataFrame(human_clue_rows).sort_values(["matchday", "name"])

pedro_pool = round_contexts[23]["coach_pool"]
pedro_matches = pedro_pool.loc[
    pedro_pool.name.astype(str).str.contains("Pedro", case=False, na=False)
]
human_coach_clue = pedro_matches[[
    "name", "credits", "expected_score", "actual_score"
]].copy()
human_coach_clue.insert(0, "matchday", 23)

print(human_sources[[
    "matchday", "manager", "winner_score", "published_lineup_evidence"
]].to_string(index=False))
print()
print(human_clues.round(2).to_string(index=False))
print()
print("MD23 human coach clue")
print(human_coach_clue.round(2).to_string(index=False))

 matchday                                 manager  winner_score                                                                                                                          published_lineup_evidence
        1 Christos Gouziotis / King of basketball        226.85                                                                                 No player names; article says every starter scored at least 25 FP.
        5              Andreas Fountas / Avengers        233.15                              Both Baldwins (Kamar and Wade); Jordan Nwora and Dan Oturu explicitly described as bench; an unnamed Olympiacos trio.
       12  Nikolaos Dimopoulos / Lion City Greens        214.30                                    Kendrick Nunn, Kenneth Faried, Ercan Osmani and Nick Weiler-Babb explicitly named; roles/captain not published.
       23  Dimitrios Kondouris / Melbourniakos 13        246.90                                                  Kameron Taylor and coach Pedro Martínez nam

In [6]:
with duckdb.connect() as connection:
    entity_names = connection.execute(
        """
        SELECT entity_id, arg_max(name, round_number) AS name
        FROM read_parquet(?)
        GROUP BY entity_id
        """,
        [str(PREDICTIONS_PATH)],
    ).fetchdf()

entity_label = dict(
    zip(entity_names.entity_id.astype(str), entity_names.name.astype(str))
)
for context in round_contexts.values():
    pool = context["coach_pool"]
    entity_label.update(
        zip(pool.coach_id.astype(str), ("Coach: " + pool.name.astype(str)))
    )

transfer_rows = []
for matchday in sorted(WORST_MATCHDAYS):
    row = backtest.loc[backtest.matchday.eq(matchday)].iloc[0]
    context = round_contexts[matchday]
    current_pool = context["player_pool"]
    current_player_entities = frozenset(
        current_pool.loc[
            current_pool.player_id.isin(context["initial"].player_ids), "entity_id"
        ].astype(str)
    )
    current_entities = current_player_entities | frozenset([context["initial"].coach_id])
    if matchday == 1:
        transfer_rows.append({
            "matchday": matchday,
            "transfers": int(row.trades),
            "trade_limit": None,
            "incoming": "Initial roster",
            "outgoing": "—",
        })
        continue
    previous = backtest.loc[backtest.matchday.eq(matchday - 1)].iloc[0]
    previous_entities = frozenset(
        [player_to_entity[player] for player in pipe_ids(previous.roster)]
        + [str(previous.coach_id)]
    )
    incoming = sorted(current_entities - previous_entities)
    outgoing = sorted(previous_entities - current_entities)
    assert len(incoming) == len(outgoing) == int(row.trades)
    transfer_rows.append({
        "matchday": matchday,
        "transfers": int(row.trades),
        "trade_limit": int(row.trade_limit),
        "incoming": "; ".join(entity_label.get(entity, entity) for entity in incoming),
        "outgoing": "; ".join(entity_label.get(entity, entity) for entity in outgoing),
    })

transfers = pd.DataFrame(transfer_rows)
print(transfers.to_string(index=False))

 matchday  transfers  trade_limit                                                                incoming                                                           outgoing
        1         11          NaN                                                          Initial roster                                                                  —
        5          3          4.0                        Devon Hall; Coach: Tomas Masiulis; Edwin Jackson              Coach: Julius Thomas; Yakuba Ouattara; Vladimir Lucic
       12          4          4.0 Trent Forrest; Mouhamet Diouf; Coach: Nenad Jakovljevic; Sasha Vezenkov      Dwayne Bacon; Shane Larkin; Coach: Sasa Obradovic; Mam Jaiteh
       23          4          4.0      Edgaras Ulanovas; Coach: Svetislav Pesic; Dan Oturu; Kendrick Nunn Coach: Giuseppe Poeta; Mbaye Ndiaye; Mckinley Wright; Jaylen Hoard
       38          4          4.0        Eugene Omoruyi; Coach: Oded Kattash; Sasha Vezenkov; Dzanan Musa   Chima Moneke; Coach: Aleksa

In [7]:
transfer_asset_rows = []
for matchday in sorted(md for md in WORST_MATCHDAYS if md > 1):
    context = round_contexts[matchday]
    pool = context["player_pool"]
    coaches = context["coach_pool"].copy()
    row = backtest.loc[backtest.matchday.eq(matchday)].iloc[0]

    current_entities = frozenset(
        pool.loc[
            pool.player_id.isin(context["initial"].player_ids), "entity_id"
        ].astype(str)
    ) | frozenset([context["initial"].coach_id])
    previous = backtest.loc[backtest.matchday.eq(matchday - 1)].iloc[0]
    previous_entities = frozenset(
        [player_to_entity[player] for player in pipe_ids(previous.roster)]
        + [str(previous.coach_id)]
    )

    for direction, entities in [
        ("IN", current_entities - previous_entities),
        ("OUT", previous_entities - current_entities),
    ]:
        for entity in sorted(entities):
            player = pool.loc[pool.entity_id.astype(str).eq(entity)]
            coach = coaches.loc[coaches.coach_id.astype(str).eq(entity)]
            if not player.empty:
                asset = player.iloc[0]
                transfer_asset_rows.append({
                    "matchday": matchday,
                    "direction": direction,
                    "name": str(asset["name"]),
                    "kind": "PLAYER",
                    "credits": float(asset.credits),
                    "predicted": float(asset.expected_fp),
                    "actual": float(asset.actual_fp),
                    "played": bool(asset.played),
                    "tail": (
                        "BELOW_P10" if asset.actual_fp < asset.p10_fp
                        else "ABOVE_P90" if asset.actual_fp > asset.p90_fp
                        else "WITHIN_P10_P90"
                    ),
                })
            elif not coach.empty:
                asset = coach.iloc[0]
                transfer_asset_rows.append({
                    "matchday": matchday,
                    "direction": direction,
                    "name": str(asset["name"]),
                    "kind": "COACH",
                    "credits": float(asset.credits),
                    "predicted": float(asset.expected_score),
                    "actual": float(asset.actual_score),
                    "played": True,
                    "tail": None,
                })
            else:
                transfer_asset_rows.append({
                    "matchday": matchday,
                    "direction": direction,
                    "name": entity_label.get(entity, entity),
                    "kind": "NOT_IN_CURRENT_MARKET",
                })

transfer_assets = pd.DataFrame(transfer_asset_rows).sort_values(
    ["matchday", "direction", "kind", "name"]
)
print(transfer_assets.round(2).to_string(index=False))

 matchday direction               name   kind  credits  predicted  actual  played           tail
        5        IN     Tomas Masiulis  COACH      6.9       9.94   -10.0    True            NaN
        5        IN         Devon Hall PLAYER      9.5      14.86    15.4    True WITHIN_P10_P90
        5        IN      Edwin Jackson PLAYER      4.4       6.50    13.0    True WITHIN_P10_P90
        5       OUT      Julius Thomas  COACH      6.4       6.74    -5.0    True            NaN
        5       OUT     Vladimir Lucic PLAYER      8.6      11.30    10.0    True WITHIN_P10_P90
        5       OUT    Yakuba Ouattara PLAYER      5.2       7.15     5.0    True WITHIN_P10_P90
       12        IN  Nenad Jakovljevic  COACH      4.8       8.07    10.0    True            NaN
       12        IN     Mouhamet Diouf PLAYER      7.4      11.81    17.6    True WITHIN_P10_P90
       12        IN     Sasha Vezenkov PLAYER     17.5      21.00     6.6    True WITHIN_P10_P90
       12        IN      Trent

## Bounded audit verification

These checks cover only the five audited rounds: round alignment, point-in-time cutoffs, initial/final roster legality, score reconstruction, attribution identities, forced-candidate feasibility, and human-source provenance.

In [8]:
target_set = set(WORST_MATCHDAYS)
assert set(attribution.matchday) == target_set
assert set(human_sources.matchday) == target_set
assert set(comparison.loc[comparison.matchday.isin(target_set), "matchday"]) == target_set
assert set(cutoff_audit.loc[cutoff_audit.matchday.isin(target_set), "matchday"]) == target_set

cutoff_subset = cutoff_audit.loc[cutoff_audit.matchday.isin(target_set)]
assert cutoff_subset["max_feature_time_strictly_before_cutoff"].all()
assert cutoff_subset["conditional_on_playing_predictions"].all()
assert cutoff_subset["actual_dnp_scored_zero_only_at_replay"].all()

for matchday, context in round_contexts.items():
    row = context["row"]
    pool = context["player_pool"]
    roster_pool = pool.loc[pool.player_id.isin(context["initial"].player_ids)]
    selected_coach = context["coach_pool"].loc[
        context["coach_pool"].coach_id.astype(str).eq(context["initial"].coach_id)
    ].iloc[0]
    player_records = {
        str(player.player_id): {
            "position": str(player.position),
            "team_id": str(player.team_id),
            "credits": float(player.credits),
        }
        for player in roster_pool.itertuples(index=False)
    }
    coach_record = {
        "team_id": str(selected_coach.team_id),
        "credits": float(selected_coach.credits),
    }
    validate_lineup(
        context["initial"], player_records, coach_record, rules,
        budget=float(row.available_budget),
    )
    validate_lineup(
        context["replay"].final_lineup, player_records, coach_record, rules,
        budget=float(row.available_budget),
    )
    assert np.isclose(context["replay"].final_score, row.actual_final_score)
    assert np.isclose(context["replay"].static_score, row.actual_static_score)

assert np.allclose(
    attribution["constrained_hindsight_oracle"] - attribution["ai_score"],
    attribution["captain_starter_hindsight_loss"]
    + attribution["selection_hindsight_loss"],
)
assert (
    attribution["constrained_hindsight_oracle"]
    <= attribution["unconstrained_hindsight_oracle"] + 1e-8
).all()
assert missed_high_scorers["legal_with_actual_constraints"].all()
assert human_sources.source_url.notna().all()
assert human_sources.source_url.str.startswith("https://www.basketstories.net/").all()

verification = pd.DataFrame([
    ["Matchday alignment", True, "Exactly MD1, MD5, MD12, MD23, MD38"],
    ["Point-in-time cutoff", True, "All feature timestamps strictly precede lock"],
    ["Roster legality", True, "Initial and replay-final lineups validated"],
    ["Replay score", True, "Static and final scores equal frozen ledger"],
    ["Attribution arithmetic", True, "Role + selection equals constrained hindsight gap"],
    ["Candidate feasibility", True, "Forced-candidate MILPs feasible under stated constraints"],
    ["Human provenance", True, "One BasketStories article URL per audited round"],
], columns=["check", "passed", "detail"])
print(verification.to_string(index=False))

                 check  passed                                                   detail
    Matchday alignment    True                       Exactly MD1, MD5, MD12, MD23, MD38
  Point-in-time cutoff    True             All feature timestamps strictly precede lock
       Roster legality    True               Initial and replay-final lineups validated
          Replay score    True              Static and final scores equal frozen ledger
Attribution arithmetic    True        Role + selection equals constrained hindsight gap
 Candidate feasibility    True Forced-candidate MILPs feasible under stated constraints
      Human provenance    True          One BasketStories article URL per audited round
